# 03 · Stage 2: one route QUBO per vehicle

**What it does.** For each group of customers produced by Stage 1, builds a small travelling-salesman QUBO
(`build_route_bqm`) that decides the visit order, with the depot fixed at the start and the end. It decodes
samples back into visit orders (`decode_route`) and runs the same brute-force formulation check.

**Why it was implemented.** Stage 1 only groups customers; the true objective (route length) needs an order.
Solving one small route per vehicle keeps each quantum run at 4–9 qubits. Variables $y_{i,t} = 1$ if
customer $i$ is visited at position $t$ ($m$ customers → $m^2$ variables):

$$E_2 = \sum_i d_{0i} y_{i,0} + \sum_{t} \sum_{i \ne j} d_{ij} y_{i,t} y_{j,t+1} + \sum_i d_{i0} y_{i,m-1}
+ P_a \sum_i \Big(\sum_t y_{i,t} - 1\Big)^2 + P_b \sum_t \Big(\sum_i y_{i,t} - 1\Big)^2$$

A valid tour is a *permutation matrix*: each customer once (penalty $P_a$), one customer per position ($P_b$).
On a valid tour both penalties are zero and $E_2$ is exactly the route length.

**Resources.**
- Code: `domains/vrp/problem.py` (`build_route_bqm`, `build_route_qubo`, `decode_route`, `route_penalty_bound`), `common/qubo_utils.py` (`verify_formulation`).
- Libraries: dimod 0.12.
- Derivation: `domains/vrp/README.md`, Stage 2 (coefficient table, tested in `tests/test_vrp_problem.py`).
- Literature: A. Lucas, *Ising formulations of many NP problems*, Front. Phys. 2014 (TSP encoding).

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
from itertools import combinations
from math import factorial

from common.qubo_utils import verify_formulation
from domains.vrp.problem import build_route_bqm, decode_route, load_instance, route_penalty_bound

inst = load_instance("data/vrp/P-n6_1.vrp")
P2 = route_penalty_bound(inst)
penalties = {"one_position": P2, "one_customer": P2}
print("penalty weight Pa = Pb =", P2, "(2 x the longest distance, 51)")

## The two routes Stage 1 produces
Flipping one bit changes at most two legs of a route, each at most 51 long, so P = 102 makes any single-bit
violation unprofitable for every possible route.

In [ ]:
reports = {}
for route in [(1, 2, 5), (3, 4)]:
    bqm = build_route_bqm(inst, route, penalties)
    reports[route] = r = verify_formulation(bqm, lambda s, route=route: decode_route(s, route) is not None)
    tours = sorted(decode_route(s, route) for s in r["ground_states"])
    print(f"route {route}: {bqm.num_variables} variables, {bqm.num_interactions} couplings, "
          f"{r['n_feasible']} valid tours of {r['n_states']} bitstrings")
    print(f"   optimal tours {tours}, length {r['ground_energy']:g}, gap {r['gap']:g}")

**Two optimal tours each.** Distances are symmetric, so a tour and its reverse have the same length. The
symmetry is left in: removing it would need extra terms, and both answers decode to the same route length.

## Decoding by variable name
`decode_route` returns the visit order only when the bits form a permutation matrix, otherwise `None`.

In [ ]:
route = (3, 4)
examples = {
    "3 then 4": {"y_3_0": 1, "y_4_0": 0, "y_3_1": 0, "y_4_1": 1},
    "both at position 0": {"y_3_0": 1, "y_4_0": 1, "y_3_1": 0, "y_4_1": 0},
    "customer 3 twice": {"y_3_0": 1, "y_4_0": 0, "y_3_1": 1, "y_4_1": 0},
}
decoded = {name: decode_route(s, route) for name, s in examples.items()}
for name, order in decoded.items():
    print(f"{name:<20} -> {order}")

## One penalty for every route Stage 1 can produce
Stage 1 always gives one 2-customer and one 3-customer group, so the check runs on all 20 such subsets.

In [ ]:
failing = []
for k in (2, 3):
    for group in combinations(inst.customers, k):
        try:
            verify_formulation(build_route_bqm(inst, group, penalties),
                               lambda s, group=group: decode_route(s, group) is not None)
        except ValueError:
            failing.append(group)
print("routes checked: 20 | failing at P = 102:", failing or "none")

## Why one-hot routes are hard for QAOA (encoding, TQ3)
Only $m!$ of the $2^{m^2}$ bitstrings are valid tours.

In [ ]:
for m in (2, 3, 4, 5):
    print(f"m = {m}: {m * m:>2} qubits, {factorial(m):>3} valid of {2 ** (m * m):>8} bitstrings "
          f"({100 * (1 - factorial(m) / 2 ** (m * m)):.1f}% infeasible)")

## Expected results
- P = 102.
- Route {1, 2, 5}: **9 variables**, 30 couplings, 6 valid tours of 512; optimal tours 1-5-2 and 2-5-1, length **45**, gap 175.
- Route {3, 4}: **4 variables**, 2 valid tours of 16; length **73**, gap 149.
- 45 + 73 = **118**, the QOPTLib optimum.
- All 20 possible routes pass at P = 102. For 3 customers 98.8% of bitstrings are infeasible.

In [ ]:
r125, r34 = reports[(1, 2, 5)], reports[(3, 4)]
assert P2 == 102
assert r125["n_vars"] == 9 and r125["n_feasible"] == 6 and r125["ground_energy"] == 45 and r125["gap"] == 175
assert r34["n_vars"] == 4 and r34["ground_energy"] == 73 and r34["gap"] == 149
assert decoded == {"3 then 4": (3, 4), "both at position 0": None, "customer 3 twice": None}
assert not failing